# Dynamic Risk Management for Trading

This notebook demonstrates how to use dynamic risk management to adaptively manage trading risk based on market conditions.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from src.risk.risk_manager import DynamicRiskManager, MarketRegime, RiskMetrics
from src.data.binance_fetcher import BinanceFetcher
from src.features.feature_engineering import FeatureEngineer

## Configuration

Load and configure risk management settings.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Risk management configuration
risk_config = {
    'var_limit': 0.02,
    'drawdown_limit': 0.20,
    'volatility_limit': 0.03,
    'volatility_window': 20,
    'trend_window': 50,
    'risk_free_rate': 0.0
}

# Initialize risk manager
risk_manager = DynamicRiskManager(
    config=risk_config,
    initial_capital=100000,
    max_position_size=1.0,
    max_leverage=3.0
)

## Data Preparation

Fetch historical data and calculate returns.

In [ ]:
# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Fetch historical data
data = await fetcher.fetch_historical_data()

# Calculate returns
returns = data['close'].pct_change().dropna()
prices = data['close']

print(f"Data points: {len(returns)}")
print(f"Mean return: {returns.mean():.4f}")
print(f"Volatility: {returns.std():.4f}")

## Risk Metrics Analysis

Calculate and analyze risk metrics.

In [ ]:
def analyze_risk_metrics(returns: pd.Series, window: int = 100):
    """Analyze risk metrics over time."""
    metrics_history = []
    
    for i in range(window, len(returns), 20):
        window_returns = returns[i-window:i]
        metrics = risk_manager.calculate_risk_metrics(
            returns=window_returns.values,
            model_confidence=0.8  # Example confidence
        )
        metrics_history.append({
            'timestamp': returns.index[i],
            'var': metrics.var,
            'es': metrics.es,
            'volatility': metrics.volatility,
            'drawdown': metrics.drawdown,
            'sharpe_ratio': metrics.sharpe_ratio
        })
    
    return pd.DataFrame(metrics_history).set_index('timestamp')

# Calculate risk metrics
risk_metrics_df = analyze_risk_metrics(returns)

# Plot risk metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# VaR and ES
axes[0,0].plot(risk_metrics_df.index, risk_metrics_df['var'], label='VaR')
axes[0,0].plot(risk_metrics_df.index, risk_metrics_df['es'], label='ES')
axes[0,0].set_title('Value at Risk and Expected Shortfall')
axes[0,0].legend()

# Volatility
axes[0,1].plot(risk_metrics_df.index, risk_metrics_df['volatility'])
axes[0,1].set_title('Volatility')

# Drawdown
axes[1,0].plot(risk_metrics_df.index, risk_metrics_df['drawdown'])
axes[1,0].set_title('Drawdown')

# Sharpe Ratio
axes[1,1].plot(risk_metrics_df.index, risk_metrics_df['sharpe_ratio'])
axes[1,1].set_title('Sharpe Ratio')

plt.tight_layout()
plt.show()

## Market Regime Analysis

Detect and analyze market regimes.

In [ ]:
def analyze_market_regimes(prices: pd.Series, returns: pd.Series, window: int = 100):
    """Analyze market regimes over time."""
    regimes = []
    
    for i in range(window, len(returns), 20):
        regime = risk_manager.detect_market_regime(
            prices=prices[:i].values,
            returns=returns[:i].values
        )
        regimes.append({
            'timestamp': returns.index[i],
            'regime': regime
        })
    
    return pd.DataFrame(regimes).set_index('timestamp')

# Analyze regimes
regime_df = analyze_market_regimes(prices, returns)

# Plot regimes
plt.figure(figsize=(15, 5))

# Plot price
plt.subplot(2, 1, 1)
plt.plot(prices.index, prices.values)
plt.title('Price')

# Plot regimes
plt.subplot(2, 1, 2)
regime_codes = {regime: i for i, regime in enumerate(MarketRegime)}
plt.plot(regime_df.index,
         [regime_codes[r] for r in regime_df['regime']])
plt.yticks(
    range(len(MarketRegime)),
    [r.value for r in MarketRegime]
)
plt.title('Market Regime')

plt.tight_layout()
plt.show()

# Print regime statistics
regime_stats = regime_df['regime'].value_counts()
print("\nRegime Statistics:")
for regime, count in regime_stats.items():
    print(f"{regime.value}: {count} ({count/len(regime_df)*100:.1f}%)")

## Position Sizing Analysis

Analyze dynamic position sizing based on market conditions.

In [ ]:
def analyze_position_sizing(risk_metrics_df: pd.DataFrame, regime_df: pd.DataFrame):
    """Analyze position sizing under different conditions."""
    positions = []
    
    for timestamp in risk_metrics_df.index:
        metrics = RiskMetrics(
            var=risk_metrics_df.loc[timestamp, 'var'],
            es=risk_metrics_df.loc[timestamp, 'es'],
            volatility=risk_metrics_df.loc[timestamp, 'volatility'],
            drawdown=risk_metrics_df.loc[timestamp, 'drawdown'],
            sharpe_ratio=risk_metrics_df.loc[timestamp, 'sharpe_ratio'],
            sortino_ratio=1.0,  # Example value
            calmar_ratio=1.0,   # Example value
            model_confidence=0.8,
            market_regime=regime_df.loc[timestamp, 'regime']
        )
        
        # Calculate position size for different predictions
        positions.append({
            'timestamp': timestamp,
            'long': risk_manager.calculate_position_size(0.5, 0.8, metrics),
            'short': risk_manager.calculate_position_size(-0.5, 0.8, metrics),
            'regime': metrics.market_regime
        })
    
    return pd.DataFrame(positions).set_index('timestamp')

# Analyze position sizing
position_df = analyze_position_sizing(risk_metrics_df, regime_df)

# Plot position sizes
plt.figure(figsize=(15, 10))

# Plot positions
plt.subplot(2, 1, 1)
plt.plot(position_df.index, position_df['long'], label='Long')
plt.plot(position_df.index, position_df['short'], label='Short')
plt.title('Position Sizes')
plt.legend()

# Plot position size by regime
plt.subplot(2, 1, 2)
regime_positions = position_df.groupby('regime').agg({
    'long': 'mean',
    'short': 'mean'
}).abs()

regime_positions.plot(kind='bar')
plt.title('Average Position Size by Regime')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Risk Limit Analysis

Analyze dynamic risk limit adjustments.

In [ ]:
def analyze_risk_limits(risk_metrics_df: pd.DataFrame):
    """Analyze risk limit adjustments."""
    limits_history = []
    initial_limits = risk_manager.get_risk_limits()
    
    for timestamp in risk_metrics_df.index:
        # Adjust limits based on performance
        risk_manager.adjust_risk_limits({
            'sharpe_ratio': risk_metrics_df.loc[timestamp, 'sharpe_ratio'],
            'calmar_ratio': 1.0,  # Example value
            'volatility': risk_metrics_df.loc[timestamp, 'volatility']
        })
        
        limits = risk_manager.get_risk_limits()
        limits['timestamp'] = timestamp
        limits_history.append(limits)
    
    return pd.DataFrame(limits_history).set_index('timestamp')

# Analyze risk limits
limits_df = analyze_risk_limits(risk_metrics_df)

# Plot risk limit adjustments
plt.figure(figsize=(15, 10))

# Plot VaR limit
plt.subplot(2, 2, 1)
plt.plot(limits_df.index, limits_df['var_limit'])
plt.title('VaR Limit')

# Plot position size limit
plt.subplot(2, 2, 2)
plt.plot(limits_df.index, limits_df['max_position_size'])
plt.title('Max Position Size')

# Plot leverage limit
plt.subplot(2, 2, 3)
plt.plot(limits_df.index, limits_df['max_leverage'])
plt.title('Max Leverage')

# Plot volatility limit
plt.subplot(2, 2, 4)
plt.plot(limits_df.index, limits_df['volatility_limit'])
plt.title('Volatility Limit')

plt.tight_layout()
plt.show()

## Performance Analysis

Analyze trading performance with risk management.

In [ ]:
def simulate_trading(position_df: pd.DataFrame, prices: pd.Series):
    """Simulate trading with risk management."""
    performance = []
    
    for timestamp in position_df.index:
        # Get position and price
        position = position_df.loc[timestamp, 'long']
        price = prices.loc[timestamp]
        
        # Update position
        metrics = risk_manager.update_position(position, price)
        metrics['timestamp'] = timestamp
        performance.append(metrics)
    
    return pd.DataFrame(performance).set_index('timestamp')

# Simulate trading
performance_df = simulate_trading(position_df, prices)

# Plot performance
plt.figure(figsize=(15, 10))

# Plot capital
plt.subplot(2, 2, 1)
plt.plot(performance_df.index, performance_df['capital'])
plt.title('Capital')

# Plot position size
plt.subplot(2, 2, 2)
plt.plot(performance_df.index, performance_df['position_size'])
plt.title('Position Size')

# Plot total P&L
plt.subplot(2, 2, 3)
plt.plot(performance_df.index, performance_df['total_pnl'])
plt.title('Total P&L')

# Plot returns
plt.subplot(2, 2, 4)
plt.plot(performance_df.index, performance_df['return'])
plt.title('Return')

plt.tight_layout()
plt.show()

# Print performance statistics
print("\nPerformance Statistics:")
print(f"Final Capital: ${performance_df['capital'].iloc[-1]:,.2f}")
print(f"Total Return: {performance_df['return'].iloc[-1]*100:.1f}%")
print(f"Max Drawdown: {performance_df['return'].min()*100:.1f}%")